# Bài 1 — Đối tượng trung tâm: `sv.Detections`

**Mục tiêu:** Hiểu cấu trúc dữ liệu quan trọng nhất — mọi thứ trong supervision xoay quanh nó.

In [2]:
!pip install -q supervision ultralytics "supervision[assets]"

## 0. Chuẩn bị (asset video + display.py )

In [3]:
from supervision.assets import download_assets, VideoAssets

download_assets(VideoAssets.VEHICLES)
print(VideoAssets.VEHICLES.value)  # "vehicles.mp4"

[2026-08-17 14:46:29] [INFO] supervision.assets.downloader - Downloading vehicles.mp4 assets


  0%|          | 0/35345757 [00:00<?, ?it/s]

vehicles.mp4


In [4]:
%%writefile display.py
# display.py — hàm hiển thị dùng chung cho toàn giáo trình
import cv2

WINDOW_NAME = "Supervision - Live"
MAX_DISPLAY_WIDTH = 1280   # thu nhỏ frame cho vừa màn hình (chỉ để XEM, không ảnh hưởng xử lý)


def show_frame(frame, window_name: str = WINDOW_NAME, wait: int = 1) -> bool:
    """Hiện frame lên cửa sổ. Trả về False nếu người dùng bấm Q/ESC (muốn thoát).

    wait=1  -> dùng cho video (hiện liên tục, không chặn)
    wait=0  -> dùng cho ảnh tĩnh (dừng lại chờ bấm phím bất kỳ)
    """
    h, w = frame.shape[:2]
    if w > MAX_DISPLAY_WIDTH:                      # thu nhỏ để vừa màn hình
        scale = MAX_DISPLAY_WIDTH / w
        frame = cv2.resize(frame, (int(w * scale), int(h * scale)))

    cv2.imshow(window_name, frame)
    key = cv2.waitKey(wait) & 0xFF
    if key in (ord("q"), ord("Q"), 27):            # Q hoặc ESC -> thoát
        return False
    return True


def close_windows():
    cv2.destroyAllWindows()

Writing display.py


## 1.1. Chạy detection đầu tiên — thấy ngay trên cửa sổ

In [12]:
import cv2
import supervision as sv
from ultralytics import YOLO

from display import show_frame, close_windows

# 1. Load model (yolov8n = nano, nhẹ nhất; tải tự động lần đầu)
model = YOLO("yolov8n.pt")

# 2. Lấy 1 frame từ video làm ảnh thử (hoặc cv2.imread("traffic.jpg"))
image = next(sv.get_video_frames_generator("vehicles.mp4"))

# 3. Chạy inference
results = model(image)[0]

# 4. ⭐ Chuyển output của YOLO thành sv.Detections
detections = sv.Detections.from_ultralytics(results)

print(detections)

# 5. 🖥️ Hiện luôn kết quả lên cửa sổ để đối chiếu với số liệu vừa in
annotated = sv.BoxAnnotator(thickness=2).annotate(image.copy(), detections)
cv2.putText(annotated, f"Phat hien: {len(detections)} doi tuong",
            (20, 60), cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 255, 0), 3)
show_frame(annotated, wait=0)   # bấm phím bất kỳ để đóng
close_windows()


0: 384x640 3 cars, 1 truck, 93.4ms
Speed: 3.3ms preprocess, 93.4ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)
Detections(xyxy=array([[     2941.1,      1269.3,      3220.8,      1500.7],
       [     944.89,      899.64,      1235.4,      1308.8],
       [     1439.8,      1077.8,      1621.3,      1231.4],
       [     1480.1,      1007.2,      1624.4,      1111.9]], dtype=float32), mask=None, confidence=array([    0.85173,     0.67523,     0.64499,     0.38049], dtype=float32), class_id=array([2, 7, 2, 2]), tracker_id=None, data={'class_name': array(['car', 'truck', 'car', 'car'], dtype='<U5')}, metadata={})


## 1.2. Mổ xẻ `sv.Detections`

Chạy đoạn sau và quan sát kỹ output (đối chiếu với các box đang thấy trên cửa sổ).

In [10]:
print("Số đối tượng:", len(detections))
print("xyxy (tọa độ box):\n", detections.xyxy)        # ndarray (N, 4)
print("confidence:", detections.confidence)             # ndarray (N,)
print("class_id:", detections.class_id)                 # ndarray (N,)
print("tên class:", detections.data["class_name"])      # ndarray (N,) dạng chuỗi
print("tracker_id:", detections.tracker_id)             # None (chưa track — Bài 5)

Số đối tượng: 4
xyxy (tọa độ box):
 [[     2941.1      1269.3      3220.8      1500.7]
 [     944.89      899.64      1235.4      1308.8]
 [     1439.8      1077.8      1621.3      1231.4]
 [     1480.1      1007.2      1624.4      1111.9]]
confidence: [    0.85173     0.67523     0.64499     0.38049]
class_id: [2 7 2 2]
tên class: ['car' 'truck' 'car' 'car']
tracker_id: None


**Cấu trúc bên trong:**

| Thuộc tính | Kiểu | Ý nghĩa |
|---|---|---|
| `xyxy` | `ndarray (N,4)` | Tọa độ `[x1, y1, x2, y2]` mỗi box |
| `confidence` | `ndarray (N,)` | Độ tin cậy 0–1 |
| `class_id` | `ndarray (N,)` | ID lớp (COCO: 2=car, 3=motorcycle, 5=bus, 7=truck) |
| `tracker_id` | `ndarray (N,)` hoặc `None` | ID theo dõi qua các frame |
| `mask` | `ndarray (N,H,W)` hoặc `None` | Mask segmentation (nếu model hỗ trợ) |
| `data` | `dict` | Metadata phụ, vd `class_name` |

> 🔑 **Tư duy cốt lõi:** `sv.Detections` giống một "bảng" NumPy — mỗi **hàng** là một đối tượng. Vì vậy nó hỗ trợ indexing/slicing y hệt NumPy (Bài 3 sẽ khai thác triệt để).

## 1.3. Model-agnostic — sức mạnh thật sự

Cùng một pipeline, chỉ đổi 1 dòng khi đổi model (chỉ để tham khảo, không cần chạy hết nếu chưa có model tương ứng):

In [11]:
# Ultralytics (YOLOv8/v9/v10/v11...)
detections = sv.Detections.from_ultralytics(results)

# Roboflow Inference (RF-DETR, SAM...)
# detections = sv.Detections.from_inference(results)

# Hugging Face Transformers (DETR...)
# detections = sv.Detections.from_transformers(results)

# NCNN, EasyOCR, MMDetection... đều có connector tương ứng

##  Checkpoint Bài 1

Cửa sổ hiện ảnh với box quanh từng xe kèm dòng chữ "Phat hien: N doi tuong"; console in được tên class của từng xe, và hai con số khớp nhau.